# Настройка DuckLake

In [1]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = True
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5
%config SqlMagic.named_parameters = "enabled"

In [2]:
import duckdb
import pandas as pd

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

Tip: You may define configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml or /Users/i.korsakov/.jupysql/config.

Did not find user configurations in /Users/i.korsakov/_code/github/pet_project_what_is_ducklake/pyproject.toml.

In [3]:
%sql duckdb:///:memory:

Connecting and switching to connection 'duckdb:///:memory:'

# Создание подключения к DuckLake

In [4]:
%%sql
INSTALL ducklake;
INSTALL postgres;

,Success


In [5]:
%%sql
CREATE OR REPLACE SECRET
(
    TYPE postgres,
    HOST 'localhost',
    PORT 5432,
    DATABASE postgres,
    USER 'postgres',
    PASSWORD 'postgres'
);

,Success


In [6]:
%%sql
CREATE OR REPLACE SECRET
(
    TYPE s3,
    URL_STYLE 'path',
    USE_SSL FALSE,
    ENDPOINT 'localhost:9000',
    KEY_ID 'minioadmin',
    SECRET 'minioadmin'
);

,Success


In [7]:
%%sql
ATTACH 'ducklake:postgres:dbname=postgres' AS my_ducklake (DATA_PATH 's3://prod/ducklake/');

,Success


In [8]:
%%sql
USE my_ducklake;

,Success


# Создание таблицы в DuckLake

In [9]:
%%sql
INSTALL fakeit FROM community;
LOAD fakeit;

CREATE OR REPLACE TABLE fake_data AS
SELECT
    s.id AS id,
    fakeit_name_full() AS name,
    fakeit_contact_email() AS email,
    fakeit_address_city() AS city,
    fakeit_address_country() AS country
FROM
    generate_series(1, 100) AS s(id);

,Success


In [10]:
%%sql
FROM fake_data

,id,name,email,city,country
0,1,Jasen Stamm,gildarunte@herzog.name,Kerlukeview,French Southern Territories
1,2,Bonita Lubowitz,glendasmitham@bernhard.com,Mafaldaport,Zambia
2,3,Yasmin Boyle,nikolashilll@kuhn.org,Gaylordtown,Indonesia
3,4,Freddie Torphy,axelgottlieb@nolan.org,Traceburgh,Libyan Arab Jamahiriya
4,5,Leilani Macejkovic,gissellejaskolski@turcotte.com,Kiehnside,Bahamas
...,...,...,...,...,...
95,96,Nathanael Jenkins,gilbertoanderson@runte.org,Terrillville,Niger
96,97,Cordelia Moore,jewelgulgowski@buckridge.io,Tillmanhaven,Palau
97,98,Lucinda Rath,titolynch@langworth.name,Flaviostad,Austria
98,99,Eliseo Schamberger,rebekahwunsch@schulist.name,Crookstown,Canada


# Изменение схемы (модели)
- [Schema Evolution](https://ducklake.select/docs/stable/duckdb/usage/schema_evolution)

In [11]:
%%sql
ALTER TABLE fake_data
ADD COLUMN name_prefix VARCHAR;

,Success


In [12]:
%%sql
FROM fake_data

,id,name,email,city,country,name_prefix
0,1,Jasen Stamm,gildarunte@herzog.name,Kerlukeview,French Southern Territories,None
1,2,Bonita Lubowitz,glendasmitham@bernhard.com,Mafaldaport,Zambia,None
2,3,Yasmin Boyle,nikolashilll@kuhn.org,Gaylordtown,Indonesia,None
3,4,Freddie Torphy,axelgottlieb@nolan.org,Traceburgh,Libyan Arab Jamahiriya,None
4,5,Leilani Macejkovic,gissellejaskolski@turcotte.com,Kiehnside,Bahamas,None
...,...,...,...,...,...,...
95,96,Nathanael Jenkins,gilbertoanderson@runte.org,Terrillville,Niger,None
96,97,Cordelia Moore,jewelgulgowski@buckridge.io,Tillmanhaven,Palau,None
97,98,Lucinda Rath,titolynch@langworth.name,Flaviostad,Austria,None
98,99,Eliseo Schamberger,rebekahwunsch@schulist.name,Crookstown,Canada,None


In [13]:
%%sql
UPDATE fake_data
SET name_prefix = fakeit_name_prefix()

,Success


In [15]:
%%sql
FROM fake_data

,id,name,email,city,country,name_prefix
0,1,Jasen Stamm,gildarunte@herzog.name,Kerlukeview,French Southern Territories,Dr.
1,2,Bonita Lubowitz,glendasmitham@bernhard.com,Mafaldaport,Zambia,Mr.
2,3,Yasmin Boyle,nikolashilll@kuhn.org,Gaylordtown,Indonesia,Ms.
3,4,Freddie Torphy,axelgottlieb@nolan.org,Traceburgh,Libyan Arab Jamahiriya,Mrs.
4,5,Leilani Macejkovic,gissellejaskolski@turcotte.com,Kiehnside,Bahamas,Miss
...,...,...,...,...,...,...
95,96,Nathanael Jenkins,gilbertoanderson@runte.org,Terrillville,Niger,Mrs.
96,97,Cordelia Moore,jewelgulgowski@buckridge.io,Tillmanhaven,Palau,Mrs.
97,98,Lucinda Rath,titolynch@langworth.name,Flaviostad,Austria,Mr.
98,99,Eliseo Schamberger,rebekahwunsch@schulist.name,Crookstown,Canada,Miss


# Time travel

In [16]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [17]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [18]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,0,"created_schema:""main""",None,None,None
1,1,"created_table:""main"".""fake_data"",inserted_into...",None,None,None
2,2,altered_table:1,None,None,None
3,3,"inserted_into_table:1,deleted_from_table:1",None,None,None


In [19]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2026-06-04 14:57:50.133936+03:00,0,1,0
1,1,2026-06-04 15:00:07.920241+03:00,1,2,1
2,2,2026-06-04 15:00:08.359207+03:00,2,2,1
3,3,2026-06-04 15:03:10.959830+03:00,2,2,2


In [20]:
%%sql
USE 'my_ducklake';

,Success


In [21]:
%%sql
SELECT * FROM fake_data AT (VERSION => 3);

,id,name,email,city,country,name_prefix
0,1,Jasen Stamm,gildarunte@herzog.name,Kerlukeview,French Southern Territories,Dr.
1,2,Bonita Lubowitz,glendasmitham@bernhard.com,Mafaldaport,Zambia,Mr.
2,3,Yasmin Boyle,nikolashilll@kuhn.org,Gaylordtown,Indonesia,Ms.
3,4,Freddie Torphy,axelgottlieb@nolan.org,Traceburgh,Libyan Arab Jamahiriya,Mrs.
4,5,Leilani Macejkovic,gissellejaskolski@turcotte.com,Kiehnside,Bahamas,Miss
...,...,...,...,...,...,...
95,96,Nathanael Jenkins,gilbertoanderson@runte.org,Terrillville,Niger,Mrs.
96,97,Cordelia Moore,jewelgulgowski@buckridge.io,Tillmanhaven,Palau,Mrs.
97,98,Lucinda Rath,titolynch@langworth.name,Flaviostad,Austria,Mr.
98,99,Eliseo Schamberger,rebekahwunsch@schulist.name,Crookstown,Canada,Miss
